# End-to-End Sparse Format Pipeline Test

This notebook tests the complete pipeline with the new sparse format:
1. Load data for multiple genomes with sparse labels and usage arrays
2. Generate train/test splits
3. Train a splice site prediction model
4. Make predictions and evaluate performance

**All labels and usage data are now stored in efficient sparse parquet format (100x smaller than dense arrays).**

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Add project to path
project_root = Path('/home/elek/projects/splicevo')
sys.path.insert(0, str(project_root / 'src'))

from splicevo.data import MultiGenomeDataLoader


/home/elek/miniforge3/envs/splicevo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Fix pyarrow extension type registration issue
import pyarrow as pa
if not hasattr(pa, '_hotfix_installed'):
    try:
        pa.unregister_extension_type("arrow.py_extension_type")
    except:
        pass
    pa._hotfix_installed = True

## Step 1: Load Data for Small Subset of Genomes

We'll load a few chromosomes from 2-3 genomes to keep this test fast.

In [3]:
# Configuration
config_file = project_root / 'configs' / 'genomes_small.json'
output_dir = Path('/tmp/splicevo_test')
output_dir.mkdir(exist_ok=True, parents=True)

# Load config
with open(config_file) as f:
    config = json.load(f)

print(f"Loaded config with {len(config['genomes'])} genomes")
for genome in config['genomes'][:3]:  # Show first 3
    print(f"  - {genome['genome_id']}:")
    print(f"     {config['usage_files'][genome['genome_id']]['tissues']}")
    print(f"     {config['usage_files'][genome['genome_id']]['timepoints']}")

Loaded config with 3 genomes
  - human_GRCh37:
     ['Brain', 'Cerebellum', 'Heart']
     [1, 5, 10]
  - mouse_GRCm38:
     ['Brain', 'Cerebellum', 'Heart']
     [1, 5, 10]
  - rat_Rnor_5.0:
     ['Brain', 'Cerebellum', 'Heart']
     [1, 5, 10]


In [4]:
# Initialize loader
loader = MultiGenomeDataLoader()

# We'll just load small genomes for this test
test_genomes = config['genomes']

print(f"\nAdding {len(test_genomes)} genomes to loader...")
for genome_config in test_genomes:
    genome_id = genome_config['genome_id']
    
    # Add genome
    loader.add_genome(
        genome_id=genome_id,
        genome_path=genome_config['genome_path'],
        gtf_path=genome_config['gtf_path'],
        chromosomes=genome_config.get('chromosomes', None),
        metadata=genome_config
    )
    print(f"\nAdded {genome_id}")
    
    # Add usage files if available
    if config['usage_files'] and genome_id in config['usage_files']:
        usage_entry = config['usage_files'][genome_id]
        usage_pattern = usage_entry.get('pattern', None)
        tissues = usage_entry.get('tissues', [])
        timepoints = usage_entry.get('timepoints', [])
        for tissue in tissues:
            for timepoint in timepoints:
                usage_file = usage_pattern.format(tissue=tissue, timepoint=timepoint)
                if os.path.exists(usage_file):
                    loader.add_usage_file(
                        genome_id=genome_id,
                        usage_file=usage_file,
                        tissue=tissue,
                        timepoint=str(timepoint)
                    )
            print(f"Added usage files for {genome_id}")



Adding 3 genomes to loader...

Added human_GRCh37
Loading usage file for human_GRCh37 - Brain 1: /home/elek/sds/sd17d003/Anamaria/spliser_for_splicevo/Homo_sapiens/Human.Brain.1.combined.tsv
Removed 1366 duplicate entries
Loaded 580739 usage entries for Brain 1
Loading usage file for human_GRCh37 - Brain 5: /home/elek/sds/sd17d003/Anamaria/spliser_for_splicevo/Homo_sapiens/Human.Brain.5.combined.tsv
Removed 1531 duplicate entries
Loaded 628274 usage entries for Brain 5
Loading usage file for human_GRCh37 - Brain 10: /home/elek/sds/sd17d003/Anamaria/spliser_for_splicevo/Homo_sapiens/Human.Brain.10.combined.tsv
Removed 1548 duplicate entries
Loaded 621477 usage entries for Brain 10
Added usage files for human_GRCh37
Loading usage file for human_GRCh37 - Cerebellum 5: /home/elek/sds/sd17d003/Anamaria/spliser_for_splicevo/Homo_sapiens/Human.Cerebellum.5.combined.tsv
Removed 1341 duplicate entries
Loaded 604212 usage entries for Cerebellum 5
Added usage files for human_GRCh37
Loading usage

In [5]:
# Load splice sites from all genomes (this may take a few minutes minutes for small subset)
print("Loading splice sites...")
loader.load_all_genomes_data(max_transcripts_per_genome=500)  # Limit to 500 transcripts for speed

# Get summary
summary = loader.get_summary()
print(f"\nLoaded {len(loader.loaded_data)} splice sites")
print("\nSummary by genome and chromosome:")
print(summary)

Loading splice sites...
Processing GTF annotations for human_GRCh37...
Loading GTF file /home/elek/sds/sd17d003/Anamaria/genomes/mazin/gtf/Homo_sapiens.gtf.gz


Loaded 701442 GTF records
Extracting transcripts...
Processing 638195 exon records from 63247 transcripts...
Created 63247 transcript objects


Processing transcripts: 100%|██████████| 1/1 [00:00<00:00, 78.55batch/s]

Collected 8154 positions (8154 positive)
Processing 8154 sequences...



Creating splice sites: 100%|██████████| 8154/8154 [00:00<00:00, 377712.49site/s]


...

Loaded 13334 examples
Processing GTF annotations for rat_Rnor_5.0...
Loading GTF file /home/elek/sds/sd17d003/Anamaria/genomes/mazin/gtf/Rattus_norvegicus.gtf.gz
Loaded 355577 GTF records
Extracting transcripts...
Processing 324158 exon records from 31419 transcripts...
Created 31419 transcript objects


Processing transcripts: 100%|██████████| 1/1 [00:00<00:00, 64.27batch/s]

Collected 11276 positions (11276 positive)
Processing 11276 sequences...



Creating splice sites: 100%|██████████| 11276/11276 [00:00<00:00, 523296.03site/s]

Loaded 11276 examples
Total loaded examples: 32764

Loaded 32764 splice sites

Summary by genome and chromosome:
      genome_id chromosome  site_type  n_sites
0  human_GRCh37         11          1      617
1  human_GRCh37         11          2      570
2  mouse_GRCm38         15          1      882
3  mouse_GRCm38         15          2      845
4  rat_Rnor_5.0         16          1      607
5  rat_Rnor_5.0         16          2      575


In [6]:
# Check available usage conditions
conditions_df = loader.get_available_conditions()
print(f"\nAvailable usage conditions: {len(conditions_df)}")
if len(conditions_df) > 0:
    print(conditions_df)
else:
    print("(No usage data loaded)")


Available usage conditions: 8
   condition_key      tissue timepoint   display_name
0        Brain_1       Brain         1        Brain 1
1        Brain_5       Brain         5        Brain 5
2       Brain_10       Brain        10       Brain 10
3   Cerebellum_5  Cerebellum         5   Cerebellum 5
4        Heart_1       Heart         1        Heart 1
5        Heart_5       Heart         5        Heart 5
6       Heart_10       Heart        10       Heart 10
7  Cerebellum_10  Cerebellum        10  Cerebellum 10


## Step 2: Extract Arrays and Save in Sparse Format

Convert loaded data to arrays with sparse usage format.

In [ ]:
# Extract sequences, labels, and usage data
# Using small window sizes for faster testing
window_size = 1000
context_size = 450
alpha_threshold = 5

print(f"Extracting arrays (window={window_size}, context={context_size})...")
print("This will save labels and usage in sparse parquet format...")

genome_dir = output_dir / 'combined_genomes'
genome_dir.mkdir(exist_ok=True)

sequences, metadata = loader.to_arrays(
    window_size=window_size,
    context_size=context_size,
    alpha_threshold=alpha_threshold,
    n_workers=4,
    use_parallel=True,
    save_memmap=genome_dir
)

print(f"\nGenerated {len(sequences)} windows")
print(f"  Sequences shape: {sequences.shape}")
print(f"  Metadata rows: {len(metadata)}")
print(f"  Labels saved in sparse format: labels.parquet")
print(f"  Usage saved in sparse format: usage.parquet")


Extracting arrays (window=1000, context=450)...
This will save labels and usage in sparse parquet format...
Building splice site index...
Indexed 4096 splice sites
Estimating window count from splice site positions...
Estimated windows (with 10% buffer): 15777
Pre-allocating memmap arrays in /tmp/splicevo_test/combined_genomes

Processing genes and writing to memmap...
Processing genome human_GRCh37...
  Processing 77 genes...
  Pre-loading genome...
  Using 4 parallel workers...


  human_GRCh37 genes: 100%|██████████| 77/77 [00:05<00:00, 13.28gene/s]


Processing genome mouse_GRCm38...
  Processing 84 genes...
  Pre-loading genome...
  Using 4 parallel workers...


  mouse_GRCm38 genes: 100%|██████████| 84/84 [00:06<00:00, 12.99gene/s]


Processing genome rat_Rnor_5.0...
  Processing 62 genes...
  Pre-loading genome...
  Using 4 parallel workers...


  rat_Rnor_5.0 genes: 100%|██████████| 62/62 [00:06<00:00, 10.23gene/s]

Saving sparse labels data (3788 entries)...
Sparse labels data saved to /tmp/splicevo_test/combined_genomes/labels_sparse.parquet
Saving sparse usage data (17849 entries)...
Sparse usage data saved to /tmp/splicevo_test/combined_genomes/usage_sparse.parquet
Trimming memmap arrays from 15777 to 1953 windows...
Flushing sequences memmap...
Memory-mapped files saved to: /tmp/splicevo_test/combined_genomes
Sparse labels statistics:
  Total entries: 3788
Sparse usage statistics:
  Total entries: 17849
  Sparsity: 0.1142%
Saved metadata to /tmp/splicevo_test/combined_genomes/metadata.csv
Created 1953 windowed examples
  Sequence shape: (1953, 1900, 4)
  Labels stored in sparse format: labels_sparse.parquet
  Usage stored in sparse format: usage_sparse.parquet
  Total donor sites: 1940
  Total acceptor sites: 1848

Generated 1953 windows
  Sequences shape: (1953, 1900, 4)
  Metadata rows: 1953
  Labels saved in sparse format: labels_sparse.parquet
  Usage saved in sparse format: usage_sparse.

In [ ]:
# Check sparse files
labels_file = genome_dir / 'labels.parquet'
usage_file = genome_dir / 'usage.parquet'

if labels_file.exists():
    labels_df = pd.read_parquet(labels_file)
    print(f"Labels sparse format: {len(labels_df)} entries")
    print(labels_df.head(10))
else:
    print("No labels file found")

if usage_file.exists():
    usage_df = pd.read_parquet(usage_file)
    print(f"\nUsage sparse format: {len(usage_df)} entries")
    print(usage_df.head(10))
else:
    print("No usage file found")


Labels sparse format: 3788 entries
   sample_idx  position  label
0           0         0      2
1           0       679      1
2           0       798      2
3           1         0      1
4           1        54      1
5           1       299      2
6           1       435      1
7           2       774      2
8           2       893      1
9           3       327      2

Usage sparse format: 17849 entries
   sample_idx  position  condition_idx  alpha  beta  sse
0           1         0              2    0.0   0.0  0.0
1           1         0              5    1.0   4.0  0.2
2           1       435              0    0.0   1.0  0.0
3           1       299              2    0.0   0.0  0.0
4           1       299              5    2.0   3.0  0.4
5           2       893              2    0.0   0.0  0.0
6           2       893              3    0.0   0.0  0.0
7           2       893              6    0.0   1.0  0.0
8           2       774              0    0.0   1.0  0.0
9           2     

### Sanity check

Sanity check that this is working as expected:

I will load original data for an exmple from output above: 
```
   sample_idx  position  condition_idx  alpha  beta  sse
1           1         0              5    1.0   4.0  0.2
4           1       299              5    2.0   3.0  0.4
```

In [9]:
from gtfparse import read_gtf

# Load splice usage file for condition 5
sse_fn = "/home/elek/sds/sd17d003/Anamaria/spliser_for_splicevo/Homo_sapiens/Human.Heart.5.combined.tsv"
sse_df = pd.read_csv(sse_fn, delimiter="\t")
sse_df

/tmp/ipykernel_1688298/393607448.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  sse_df = pd.read_csv(sse_fn, delimiter="\t")


,Sample,Region,Site,Strand,Gene,SSE,alpha_count,beta1_count,beta2_count,MultiGeneFlag,Others,Partners,Competitors
0,1443sTSm.Human.Heart.CS23.Male,1,12722,+,NaN,0.0,0,1,0,False,[],{13220: 0},[13052]
1,1443sTSm.Human.Heart.CS23.Male,1,13053,+,NaN,1.0,2,0,0,False,[],{13220: 2},[12721]
2,1443sTSm.Human.Heart.CS23.Male,1,13220,+,NaN,0.4,2,3,0,False,[],"{13052: 2, 12721: 0}",[]
3,1443sTSm.Human.Heart.CS23.Male,1,14669,-,NaN,0.0,0,128,0,False,[],{14969: 0},"[14737, 14829]"
4,1443sTSm.Human.Heart.CS23.Male,1,14738,-,NaN,0.0,0,199,21,False,[],{14969: 0},"[14668, 14829]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
642781,1443sTSm.Human.Heart.CS23.Male,Y,59356944,+,NaN,1.0,13,0,0,False,"[59342014, 59363482]",{59357702: 13},[]
642782,1443sTSm.Human.Heart.CS23.Male,Y,59357702,+,NaN,1.0,13,0,0,False,"[59342014, 59363482]",{59356943: 13},[]
642783,1443sTSm.Human.Heart.CS23.Male,Y,59357772,+,NaN,1.0,1,0,0,False,"[59342014, 59363482]",{59357911: 1},[]
642784,1443sTSm.Human.Heart.CS23.Male,Y,59357911,+,NaN,1.0,1,0,0,False,"[59342014, 59363482]",{59357771: 1},[]


I will pull up metadata info for this sequence (`sample_idx=1`)

In [10]:
metadata.iloc[1]

genome_id           human_GRCh37
chromosome                    11
gene_id                hum.10002
strand                         +
window_start            18433956
window_end              18434956
n_donor_sites                  4
n_acceptor_sites               2
Name: 1, dtype: object

In [11]:
sse_df[(sse_df['Region'] == 11) & (sse_df['Site'] > 18433955) & (sse_df['Site'] < 18434955)]

,Sample,Region,Site,Strand,Gene,SSE,alpha_count,beta1_count,beta2_count,MultiGeneFlag,Others,Partners,Competitors
343239,1443sTSm.Human.Heart.CS23.Male,11,18433956,+,NaN,0.2,1,0,4,False,"[18362957, 18425335, 18427119, 18472509, 18500252, 18624704]",{18434255: 1},[18428885]
343240,1443sTSm.Human.Heart.CS23.Male,11,18434255,+,NaN,0.4,2,0,3,False,"[18362957, 18425335, 18427119, 18472509, 18500252, 18624704]","{18428885: 1, 18433955: 1}",[]


***

## Step 3: Generate Train/Test Splits

Split data by chromosome. For simplicity, here we ignore psecies info.

In [12]:
# For testing, we'll do a simple split without orthology
# Just split by sample indices (80/20 split)

n_samples = len(sequences)
n_train = int(0.8 * n_samples)

# Random shuffle for split
np.random.seed(42)
indices = np.random.permutation(n_samples)
train_indices = indices[:n_train]
test_indices = indices[n_train:]

print(f"Splitting {n_samples} samples:")
print(f"  Train: {len(train_indices)} ({100*len(train_indices)/n_samples:.1f}%)")
print(f"  Test: {len(test_indices)} ({100*len(test_indices)/n_samples:.1f}%)")

Splitting 1953 samples:
  Train: 1562 (80.0%)
  Test: 391 (20.0%)


In [13]:
# Create split directories
train_dir = output_dir / 'train'
test_dir = output_dir / 'test'
train_dir.mkdir(exist_ok=True)
test_dir.mkdir(exist_ok=True)

# Save train sequences
print("\nSaving train split...")
train_seq_mmap = np.memmap(train_dir / 'sequences.mmap', dtype='float32', mode='w+', 
                           shape=(len(train_indices), sequences.shape[1], 4))
train_seq_mmap[:] = sequences[train_indices]
train_seq_mmap.flush()
del train_seq_mmap

# Save train metadata
train_metadata = metadata.iloc[train_indices].copy()
train_metadata = train_metadata.reset_index(drop=True)
train_metadata.to_csv(train_dir / 'metadata.csv', index=False)

print(f"Train sequences: {train_dir / 'sequences.mmap'}")



Saving train split...
Train sequences: /tmp/splicevo_test/train/sequences.mmap


In [14]:
# Save test sequences
print("\nSaving test split...")
test_seq_mmap = np.memmap(test_dir / 'sequences.mmap', dtype='float32', mode='w+',
                          shape=(len(test_indices), sequences.shape[1], 4))
test_seq_mmap[:] = sequences[test_indices]
test_seq_mmap.flush()
del test_seq_mmap

# Save test metadata
test_metadata = metadata.iloc[test_indices].copy()
test_metadata = test_metadata.reset_index(drop=True)
test_metadata.to_csv(test_dir / 'metadata.csv', index=False)

print(f"Test sequences: {test_dir / 'sequences.mmap'}")



Saving test split...
Test sequences: /tmp/splicevo_test/test/sequences.mmap


In [ ]:
# Split and save sparse labels and usage data
labels_file = genome_dir / 'labels.parquet'
usage_file = genome_dir / 'usage.parquet'

# Remap sample indices
train_idx_map = {old: new for new, old in enumerate(train_indices)}
test_idx_map = {old: new for new, old in enumerate(test_indices)}

# Split labels
if labels_file.exists():
    print("\nSplitting sparse labels data...")
    labels_df = pd.read_parquet(labels_file)
    
    # Filter and remap for train
    train_labels = labels_df[labels_df['sample_idx'].isin(train_indices)].copy()
    train_labels['sample_idx'] = train_labels['sample_idx'].map(train_idx_map)
    train_labels.to_parquet(train_dir / 'labels.parquet', compression='snappy', index=False)
    print(f"  Train labels: {len(train_labels)} entries")
    
    # Filter and remap for test
    test_labels = labels_df[labels_df['sample_idx'].isin(test_indices)].copy()
    test_labels['sample_idx'] = test_labels['sample_idx'].map(test_idx_map)
    test_labels.to_parquet(test_dir / 'labels.parquet', compression='snappy', index=False)
    print(f"  Test labels: {len(test_labels)} entries")
else:
    print("\n  No labels data to split")

# Split usage
if usage_file.exists():
    print("\nSplitting sparse usage data...")
    usage_df = pd.read_parquet(usage_file)
    
    # Filter and remap for train
    train_usage = usage_df[usage_df['sample_idx'].isin(train_indices)].copy()
    train_usage['sample_idx'] = train_usage['sample_idx'].map(train_idx_map)
    train_usage.to_parquet(train_dir / 'usage.parquet', compression='snappy', index=False)
    print(f"  Train usage: {len(train_usage)} entries")
    
    # Filter and remap for test
    test_usage = usage_df[usage_df['sample_idx'].isin(test_indices)].copy()
    test_usage['sample_idx'] = test_usage['sample_idx'].map(test_idx_map)
    test_usage.to_parquet(test_dir / 'usage.parquet', compression='snappy', index=False)
    print(f"  Test usage: {len(test_usage)} entries")
else:
    print("\n  No usage data to split")



Splitting sparse labels data...
  Train labels: 2971 entries
  Test labels: 817 entries

Splitting sparse usage data...
  Train usage: 13890 entries
  Test usage: 3959 entries


In [16]:
# Save metadata.json for each split
train_meta = {
    'sequences_shape': [len(train_indices), sequences.shape[1], 4],
    'sequences_dtype': 'float32',
    'labels_format': 'sparse',
    'labels_dtype': 'int8',
    'window_size': window_size,
    'context_size': context_size,
    'n_conditions': len(conditions_df),
    'usage_format': 'sparse',
    'usage_conditions': list(conditions_df['condition_key']) if len(conditions_df) > 0 else []
}

test_meta = {
    'sequences_shape': [len(test_indices), sequences.shape[1], 4],
    'sequences_dtype': 'float32',
    'labels_format': 'sparse',
    'labels_dtype': 'int8',
    'window_size': window_size,
    'context_size': context_size,
    'n_conditions': len(conditions_df),
    'usage_format': 'sparse',
    'usage_conditions': list(conditions_df['condition_key']) if len(conditions_df) > 0 else []
}

with open(train_dir / 'metadata.json', 'w') as f:
    json.dump(train_meta, f, indent=2)

with open(test_dir / 'metadata.json', 'w') as f:
    json.dump(test_meta, f, indent=2)

print("Splits created successfully with sparse format")
print(f"  Train dir: {train_dir}")
print(f"  Test dir: {test_dir}")


Splits created successfully with sparse format
  Train dir: /tmp/splicevo_test/train
  Test dir: /tmp/splicevo_test/test


### Sanity check

Test that train/test split data is the same as loaded genome data

Load genome data

In [17]:
from splicevo.data.data_loader import load_memmap_data

data_dir="/home/elek/sds/sd17d003/Anamaria/splicevo/data_new/processed_small_5kb/"

sequences = {}
labels = {}
usage = {}
metadata = {}
meta = {}
for genome in ['mouse_GRCm38', 'human_GRCh37']:
    genome_dir = os.path.join(data_dir, genome)
    genome_sequences, genome_labels, genome_usage, genome_metadata = load_memmap_data(
        genome_dir,
        load_labels=True,
        load_usage=True
    )
    json_fn = os.path.join(genome_dir, "metadata.json")
    with open(json_fn, "r") as f:
        genome_meta = json.load(f)

    sequences[genome] = genome_sequences
    labels[genome] = genome_labels
    usage[genome] = genome_usage
    metadata[genome] = genome_metadata
    meta[genome] = genome_meta


Load test split

In [18]:
from splicevo.data.data_loader import load_memmap_data

test_dir="/home/elek/sds/sd17d003/Anamaria/splicevo/data_new/splits_small_5kb/mouse_human/test"

test_sequences, test_labels, test_usage, test_metadata = load_memmap_data(
    test_dir,
    load_labels=True,
    load_usage=True
)

json_fn = os.path.join(test_dir, "metadata.json")
with open(json_fn, "r") as f:
    test_meta = json.load(f)


I want to check if any given sequence has correctly matched conditions in original loaded data and in test split.

The conditions in the test split are:

In [19]:
seq_usage = test_usage[(test_usage['sample_idx'] == 2) & (test_usage['position'] == 501)].sort_values('condition_idx')
seq_usage

,sample_idx,position,condition_idx,alpha,beta,sse
31,2,501,0,275.0,2.0,0.993
32,2,501,1,255.0,3.0,0.988
33,2,501,2,275.0,2.0,0.993
34,2,501,3,268.0,1.0,0.996
35,2,501,5,298.0,2.0,0.993
36,2,501,6,310.0,4.0,0.987
37,2,501,7,136.0,0.0,1.000


In [20]:
test_conds = test_meta['usage_condition_mapping']
for i in seq_usage['condition_idx']:
    cond = test_conds[str(i)]
    print(f"{cond['condition_key']}: {seq_usage[seq_usage['condition_idx']==i]['sse'].values[0]}")

Brain_1: 0.993
Brain_5: 0.988
Brain_10: 0.993
Cerebellum_5: 0.996
Heart_1: 0.993
Heart_5: 0.987
Heart_10: 1.0


Find coordinates for this sequence

In [21]:
chr = test_metadata.iloc[2]['chromosome']
wst = test_metadata.iloc[2]['window_start']
wen = test_metadata.iloc[2]['window_end']
std = test_metadata.iloc[2]['strand']
print(f"chr {chr}: {wst}-{wen} ({std})")

chr 22: 36907048-36912048 (-)


Find index of the same sequence in loaded data

In [22]:
load_metdata = metadata['human_GRCh37']
seq_metadata = load_metdata[(load_metdata['chromosome']==chr) & (load_metdata['window_start']==wst) & \
                            (load_metdata['chromosome']==chr) & (load_metdata['window_end']==wen) & \
                            (load_metdata['strand']==std)]

seq_metadata

,genome_id,chromosome,gene_id,strand,window_start,window_end,n_donor_sites,n_acceptor_sites
16166,human_GRCh37,22,hum.42250,-,36907048,36912048,8,13


In [23]:
load_usage = usage['human_GRCh37']
seq_load_usage = load_usage[(load_usage['sample_idx']==seq_metadata.index[0]) & (load_usage['position']==501)]
seq_load_usage

,sample_idx,position,condition_idx,alpha,beta,sse
339991,16166,501,0,275.0,2.0,0.993
339992,16166,501,1,255.0,3.0,0.988
339993,16166,501,2,275.0,2.0,0.993
339994,16166,501,3,268.0,1.0,0.996
339995,16166,501,4,298.0,2.0,0.993
339996,16166,501,5,310.0,4.0,0.987
339997,16166,501,6,136.0,0.0,1.000


In [24]:
load_conds = meta['human_GRCh37']['usage_condition_mapping']
load_conds
for i in seq_load_usage['condition_idx']:
    cond = load_conds[str(i)]
    print(f"{cond['condition_key']}: {seq_load_usage[seq_load_usage['condition_idx']==i]['sse'].values[0]}")

Brain_1: 0.993
Brain_5: 0.988
Brain_10: 0.993
Cerebellum_5: 0.996
Heart_1: 0.993
Heart_5: 0.987
Heart_10: 1.0


## Step 4: Load Data with Sparse Usage Reconstruction

Test loading the split data and reconstructing dense usage arrays per batch.